# 04 — Regex (`re` module)

Pattern matching for log parsing, validation, and extraction. The functions and patterns you reach for most.

## Pattern cheat-sheet

| Pattern | Matches |
|---|---|
| `.` | any char (except newline) |
| `\d` `\D` | digit / non-digit |
| `\w` `\W` | word char [a-zA-Z0-9_] / non-word |
| `\s` `\S` | whitespace / non-whitespace |
| `^` `$` | start / end of line |
| `*` `+` `?` | 0+, 1+, 0-or-1 |
| `{2,4}` | 2 to 4 repetitions |
| `[abc]` `[^abc]` | char in set / not in set |
| `(...)` | capturing group |
| `(?:...)` | non-capturing group |
| `(?P<name>...)` | named group |
| `a|b` | a or b |

Use **raw strings** `r"..."` for patterns so backslashes don't get mangled.

## The core functions

In [ ]:
import re

text = "error code 503 on host web-01"

# search: find first match anywhere -> Match object or None
m = re.search(r"\d+", text)
print(m.group())          # '503'
print(m.start(), m.end()) # position

# match: must match at the START of the string
print(re.match(r"error", text))      # matches
print(re.match(r"host", text))       # None

# findall: every match -> list
print(re.findall(r"\d+", "a1 b22 c333"))   # ['1', '22', '333']

# sub: replace
print(re.sub(r"\d+", "X", text))     # error code X on host web-X

# split on a pattern
print(re.split(r"\s+", "a   b\t c"))  # ['a', 'b', 'c']

## Capturing groups — extract structured data

In [ ]:
import re

line = "2026-06-15 10:00:02 ERROR disk full on /var"

# groups by position
m = re.search(r"(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (\w+) (.+)", line)
print(m.group(1))   # date
print(m.group(2))   # time
print(m.group(3))   # level
print(m.group(4))   # message
print(m.groups())   # all as a tuple

# named groups are clearer
pat = r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\S+) (?P<level>\w+) (?P<msg>.+)"
m = re.search(pat, line)
print(m.group("level"), "->", m.group("msg"))
print(m.groupdict())   # dict of all named groups

## Compile once, reuse (faster in loops)

In [ ]:
import re

ip_re = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

logs = [
    "request from 10.0.0.5 ok",
    "denied 192.168.1.100",
    "no ip here",
]
for line in logs:
    m = ip_re.search(line)
    if m:
        print(line, "=> IP:", m.group())

# findall with the compiled pattern
blob = "hits: 10.0.0.1, 10.0.0.2, 10.0.0.2"
print(set(ip_re.findall(blob)))   # unique IPs

## Useful flags

In [ ]:
import re

# IGNORECASE
print(re.findall(r"error", "ERROR Error error", re.IGNORECASE))

# MULTILINE: ^ and $ match each line
text = "web-01 up\nweb-02 down\ndb-01 up"
print(re.findall(r"^(\S+) down$", text, re.MULTILINE))

# DOTALL: . also matches newlines
print(re.search(r"start.*end", "start\nmiddle\nend", re.DOTALL).group())

## Handy ready-made patterns

In [ ]:
import re

patterns = {
    "ipv4":      r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
    "email":     r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "url":       r"https?://[^\s]+",
    "timestamp": r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}",
    "http_code": r"\b[1-5]\d{2}\b",
    "uuid":      r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}",
    "duration_ms": r"(\d+)ms",
}

sample = "2026-06-15T10:00:02 GET https://api.svc/health 200 from 10.0.0.5 in 42ms id=550e8400-e29b-41d4-a716-446655440000"
for name, pat in patterns.items():
    found = re.findall(pat, sample)
    print(f"{name:12} -> {found}")

## Real-world: parse an nginx-style access log

In [ ]:
import re
from collections import Counter

log_lines = [
    '10.0.0.5 - - [15/Jun/2026:10:00:01] "GET /health HTTP/1.1" 200 12',
    '10.0.0.6 - - [15/Jun/2026:10:00:02] "POST /login HTTP/1.1" 500 88',
    '10.0.0.5 - - [15/Jun/2026:10:00:03] "GET /api/v1 HTTP/1.1" 404 0',
]

pat = re.compile(
    r'(?P<ip>\S+) .* "(?P<method>\S+) (?P<path>\S+) [^"]+" (?P<status>\d+) (?P<bytes>\d+)'
)

status_counts = Counter()
for line in log_lines:
    m = pat.search(line)
    if m:
        d = m.groupdict()
        status_counts[d["status"]] += 1
        if d["status"].startswith("5"):
            print(f"5xx! {d['ip']} {d['method']} {d['path']}")

print("status counts:", dict(status_counts))